In [12]:
%pip install torch

import torch
import torch.nn as nn
from torch.nn import functional as F


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
torch.manual_seed(2077)

B,T,C = 2, 4, 4 # batch, time, channels
heads_size = 2
num_heads = C // heads_size
x = torch.randn(B,T,C)

W_Q = nn.Linear(C, C, bias=False)
W_K = nn.Linear(C, C, bias=False)
W_V = nn.Linear(C, C, bias=False)
W_O = nn.Linear(C, C, bias=False )

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,  K.transpose(-2, -1)) / (d_k ** 0.5)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights


def multi_head_attention(x, W_Q, W_K, W_V, W_O, num_heads):
    B, T, C = x.shape
    head_size = C // num_heads
    
    # (B, T, C)
    Q = W_Q(x)
    K = W_K(x)
    V = W_V(x)
    
    # (B, T, C) -> (B, T, num_heads, head_size) -> (B, num_heads, T, head_size)
    Q = Q.view(B, T, num_heads, head_size).transpose(1, 2)
    K = K.view(B, T, num_heads, head_size).transpose(1, 2)
    V = V.view(B, T, num_heads, head_size).transpose(1, 2)
    
    attended_values, attention_weights = scaled_dot_product_attention(Q, K, V)
    
    output = attended_values.transpose(1, 2).contiguous().view(B, T, C)
    
    output = W_O(output)
    
    return output, attention_weights

output, attn_weights = multi_head_attention(x, W_Q, W_K, W_V, W_O, num_heads)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")



Input shape: torch.Size([2, 4, 4])
Output shape: torch.Size([2, 4, 4])
Attention weights shape: torch.Size([2, 2, 4, 4])


In [ ]:
print(x)

net = nn.Sequential(
    nn.Linear(C, 4 * C),
    nn.ReLU(),
    nn.Linear(4 * C, C),
    # nn.Dropout(0.1) Pour eviter le sur-apprentissage
)

def feed_forward(x):
    return net(x)

feed_forward(x)

tensor([[[-0.1370, -0.0994,  0.9848,  0.5551],
         [ 0.0283, -1.4808,  2.3762, -1.8159],
         [ 0.1494, -0.3934,  1.2911, -0.3457],
         [-2.5864, -0.5203,  0.0598,  1.1132]],

        [[-1.0157, -1.4712, -0.3407,  0.5730],
         [ 0.1688, -1.5160,  1.1371,  0.8592],
         [-0.6475,  0.9308, -1.1880,  0.9021],
         [ 1.4104, -0.0855, -1.6398, -2.3788]]])


tensor([[[-0.3501, -0.2433, -0.0141,  0.1703],
         [-0.1461, -0.2953, -0.0497,  0.1245],
         [-0.1975, -0.1950, -0.0618,  0.1391],
         [-0.0543, -0.2545, -0.0142, -0.1634]],

        [[ 0.1178, -0.1648, -0.1044,  0.1414],
         [-0.2143, -0.1901, -0.0335,  0.2227],
         [-0.2872, -0.4147, -0.0067, -0.3824],
         [ 0.0013,  0.1450,  0.0364, -0.0386]]], grad_fn=<ViewBackward0>)